## Question

One hundred prisoners are isolated. Each day, one prisoner is randomly selected to visit a room with a light switch, initially off, and may flip it or leave it unchanged. Once they decide everyone has been to the room at least once, they can leave if they are correct or everyone dies if they are incorrect. They can agree on a strategy beforehand but cannot communicate afterward.

Can they safely announce that everyone has visited, and guarantee that they eventually do?

## Protocol

Choose one prisoner as the counter. Each of the other 99 turns the switch on the first time they find it off, and never signals again. If an uncounted prisoner finds the switch on, they leave it alone and may signal on a later visit. The counter turns every on switch off and adds one to their count. When the counter reaches 99, every non-counter has signaled, so the counter can announce that everyone has visited. Under the assumed random selection of a prisoner each day, this happens with probability 1, though there is no fixed deadline by which it must happen.

## Expected time

With $k$ non-counter prisoners still needing to signal and the switch off, the expected wait for an uncounted prisoner to signal is $100/k$ days. Once the switch is on, the expected wait for the counter to visit and clear it is 100 days. Summing these waits for $k=99,98,\ldots,1$ gives

$\mathbb{E}[T_{100}]=100\sum_{k=1}^{99}\frac1k+100(99)=100H_{99}+9900\approx 10{,}417.74\text{ days}$,

or about 28.5 years. This is the expected completion time for the agreed protocol, not a guarantee of completion by a particular date. The simulation below estimates the probability that the protocol has finished by selected deadlines.

## Python simulation


In [ ]:
import random
import matplotlib.pyplot as plt


rng = random.Random(0)


def simulate_protocol(prisoners_count):
    counter = 0
    signaled = [False] * prisoners_count
    switch_on = False
    signals_counted = 0
    days = 0

    while signals_counted < prisoners_count - 1:
        selected = rng.randrange(prisoners_count)
        days += 1

        if selected == counter:
            if switch_on:
                switch_on = False
                signals_counted += 1
        elif not signaled[selected] and not switch_on:
            signaled[selected] = True
            switch_on = True

    return days


prisoners_count = 100
days_to_check = list(range(2_000, 20_001, 2_000))
simulations = 2_000

# Each run follows the agreed light-switch protocol through completion.
completion_days = [simulate_protocol(prisoners_count) for _ in range(simulations)]
probabilities = [
    sum(completion_day <= days for completion_day in completion_days) / simulations
    for days in days_to_check
]
expected_days = prisoners_count * sum(
    1 / k for k in range(1, prisoners_count)
) + prisoners_count * (prisoners_count - 1)

print(f"Simulated mean completion time: {sum(completion_days) / simulations:.1f} days")
print(f"Exact expected completion time: {expected_days:.2f} days")
print("Probability the protocol has finished by each deadline")
for days, probability in zip(days_to_check, probabilities):
    print(f"{days:>5} days: {probability:.3f}")

plt.figure(figsize=(10, 4))
plt.plot(days_to_check, probabilities, marker="o", linewidth=2)
plt.xlabel("Days")
plt.ylabel("Probability protocol has finished")
plt.title(f"Light-switch protocol completion ({simulations:,} simulations)")
plt.ylim(0, 1.05)
plt.xticks(days_to_check)
plt.grid(axis="y", alpha=0.25)
plt.show()